# Field Shift — site climate for Nakasongola

One notebook for one question: **what do I plant, and when**, for Nakasongola
district (1.35 N, 32.45 E).

This file is the data layer. It pulls NASA POWER daily data and a SoilGrids
point extract, builds the monthly rainfall climatology, detects a planting onset
per season per year, and writes **`results/site_climate.json`** plus one figure.
Crop windows and `app_data.json` come in later cells as the rules table lands.

Run top to bottom. On a clean runtime the first code cell downloads the raw
cache; everything after that works offline.

**One assumption, stated once:** planting is possible when a crop's rainfall
threshold is met over a rolling 10-day sum and root-zone wetness (GWETROOT) is
at or above 0.40. It is our assumption, not a sourced agronomic rule. Until
`rules/rules.csv` has crop thresholds, the site screens at 20 mm per 10 days and
the sensitivity to 15 and 25 mm is reported next to it.


In [1]:
import sys
from pathlib import Path

import pandas as pd

NOTEBOOKS = Path.cwd()
if (NOTEBOOKS / "field_shift_data.py").exists():
    sys.path.insert(0, str(NOTEBOOKS))
    ROOT = NOTEBOOKS.parent
else:  # run from the repo root
    sys.path.insert(0, str(NOTEBOOKS / "notebooks"))
    ROOT = NOTEBOOKS

import field_shift_data as fs

pd.set_option("display.width", 120)
print("repo root:", ROOT)
print("site:", fs.SITE_NAME, fs.LAT, "N", fs.LON, "E", "| period:", fs.START, "to", fs.END)

repo root: /var/lib/openbot/OpenBot/Agents/agent-9f820081-c613-4ab3-b8ba-2d5feec5639d/field-shift
site: Nakasongola 1.35 N 32.45 E | period: 1991-01-01 to 2025-12-31


## 1. Raw data

POWER daily point data and the SoilGrids extract, cached in `data/raw/` (not in git).

In [2]:
sys.path.insert(0, str(ROOT / "scripts"))
import download_data

download_data.get_power(force=False)
download_data.get_soilgrids(force=False)

POWER: cached at data/raw/power_daily_1.35N_32.45E_1991-2025.csv (325436 B)
SoilGrids: cached at data/raw/soilgrids_point_1.35N_32.45E.json


PosixPath('/var/lib/openbot/OpenBot/Agents/agent-9f820081-c613-4ab3-b8ba-2d5feec5639d/field-shift/data/raw/soilgrids_point_1.35N_32.45E.json')

## 2. Load and check

`-999` is POWER's missing-data sentinel and is converted to NaN, then counted.
Missing days, duplicates and leftover NaNs are reported rather than filled.

In [3]:
power = fs.read_power_csv()
daily = power.daily
print(f"rows: {len(daily)}  {daily.index.min().date()} to {daily.index.max().date()}")
display(pd.DataFrame([power.quality]).T.rename(columns={0: "value"}))
display(daily.describe().T)

rows: 12784  1991-01-01 to 2025-12-31


,value
days_expected,12784
days_present,12784
duplicate_dates,0
missing_dates_count,0
missing_dates_sample,[]
sentinel_counts,"{'PRECTOTCORR': 0, 'T2M': 0, 'GWETROOT': 0}"
sentinel_value,-999.0
nan_counts,"{'rain_mm': 0, 't2m_c': 0, 'gwetroot': 0}"


,count,mean,std,min,25%,50%,75%,max
rain_mm,12784.0,3.050980,4.378241,0.00,0.33,1.56,3.9625,64.32
t2m_c,12784.0,24.470221,1.840887,18.99,23.13,24.24,25.7100,31.16
gwetroot,12784.0,0.503902,0.048216,0.44,0.47,0.49,0.5300,0.72


## 3. Monthly rainfall climatology

Whole calendar months only: a month counts when every day is present and no rain value is missing.

In [4]:
totals = fs.monthly_total_series(daily)
monthly, annual = fs.monthly_climatology(totals)
display(pd.DataFrame(monthly).set_index("month"))
print("annual:")
display(pd.Series(annual))

,rain_mm_mean,rain_mm_median,rain_mm_p25,rain_mm_p75,rain_mm_min,rain_mm_max,n_years,t2m_mean_c,gwetroot_mean
month,,,,,,,,,
1,38.1,33.5,15.3,53.7,1.1,111.4,35,25.83,0.474
2,33.2,28.8,14.7,44.4,0.5,97.0,35,26.84,0.461
3,90.3,82.7,62.4,108.6,18.0,258.1,35,26.38,0.475
4,138.9,135.0,102.0,160.6,54.6,322.8,35,24.53,0.511
5,135.0,120.9,96.7,153.6,68.2,310.2,35,23.36,0.530
6,80.8,76.8,48.2,100.3,20.5,190.6,35,23.06,0.511
7,77.0,66.5,55.6,93.1,9.6,175.9,35,23.06,0.501
8,104.9,108.7,84.1,130.4,23.0,182.1,35,23.54,0.510
9,114.0,116.0,78.2,145.7,12.5,178.8,35,24.02,0.516


annual:


n_years_complete      35.0
rain_mm_mean        1114.4
rain_mm_median      1074.8
rain_mm_min          650.8
rain_mm_max         1595.5
dtype: float64

In [5]:
fig_path = fs.plot_monthly_climatology(monthly)
print("wrote", fig_path.relative_to(ROOT))

wrote results/fig_monthly_rain_climatology.png


## 4. Onset per season per year

Onset is the first day the trailing 10-day rain sum crosses the threshold while
GWETROOT is at or above 0.40, **and** the previous 10-day sum was below it. The
crossing condition is what stops a search window that opens mid-season from
returning the window's first day every year.

Note the soil gate: GWETROOT never dropped below 0.40 in this record, so at this
site the soil condition never binds and onset is decided by rainfall alone.

In [6]:
onsets = fs.onset_table(daily)
onset_df = pd.DataFrame(onsets)
display(onset_df.pivot(index="year", columns="season", values="onset_date").tail(10))
display(pd.DataFrame(fs.onset_summary(onsets, daily)).set_index("season"))

season,first_rains,second_rains
year,,
2016,2016-03-09,2016-08-06
2017,2017-02-01,2017-08-01
2018,2018-02-25,2018-08-06
2019,2019-02-21,2019-08-04
2020,2020-02-01,2020-08-21
2021,2021-02-01,2021-09-06
2022,2022-03-04,2022-08-01
2023,2023-03-14,2023-08-03
2024,2024-02-12,2024-08-01


,window_start_md,window_end_md,years_evaluated,years_found,onset_doy_median,onset_doy_p25,onset_doy_p75,onset_doy_min,onset_doy_max,years_with_dry_spell_ge_7d_in_next_30d,threshold_sensitivity
season,,,,,,,,,,,
first_rains,02-01,05-31,35,35,53.0,41.5,73.5,32,97,17,"[{'rain_10d_threshold_mm': 15.0, 'years_found'..."
second_rains,08-01,11-30,35,35,219.0,214.0,231.0,213,301,4,"[{'rain_10d_threshold_mm': 15.0, 'years_found'..."


## 5. Soil at the point (SoilGrids, 0–30 cm depth-weighted)

In [7]:
soil = fs.read_soilgrids()
display(pd.Series(soil["summary_0_30cm"]))
print("texture sum (%):", soil["texture_sum_pct"], "| source file:", soil["source_file"])

ph_h2o       6.12
clay_pct    38.57
sand_pct    39.63
silt_pct    21.80
dtype: float64

texture sum (%): 100.0 | source file: data/raw/soilgrids_point_1.35N_32.45E.json


## 6. Write the output

The JSON carries its own provenance: source URLs, the raw-file SHA-256, the assumption list, and the data-quality counts.

In [8]:
payload = fs.build_site_climate()
out = fs.write_outputs(payload)
print("wrote", out.relative_to(ROOT))
print("sha256 of raw cache:", payload["meta"]["raw_cache"]["sha256"])
print("bytes:", out.stat().st_size)

wrote results/site_climate.json
sha256 of raw cache: ff9cec9c6582d8532e3140d213e40ce3ff344b0ca4ffbc22cb150929e21e0483
bytes: 39475
